In [1]:
import os, sys, random, shutil, warnings, glob, time, itertools, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
from torchvision.utils import save_image
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from skimage.metrics import structural_similarity as ssim, peak_signal_noise_ratio as psnr
import subprocess
from concurrent.futures import ThreadPoolExecutor
import pickle
import gc
warnings.filterwarnings('ignore')

OUTPUT_DIR = '/kaggle/working'
CKPT_DIR   = os.path.join(OUTPUT_DIR, 'checkpoints')
IMG_DIR    = os.path.join(OUTPUT_DIR, 'images')
for d in [CKPT_DIR, IMG_DIR]:
    os.makedirs(d, exist_ok=True)

IMG_SIZE     = 128
BATCH        = 8
LR           = 0.0002
BETAS        = (0.5, 0.999)
EPOCHS       = 25
LAMBDA_CYCLE = 10.0
LAMBDA_ID    = 0.5
N_RES        = 6
USE_N        = 2500

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
n_gpus  = torch.cuda.device_count()
print(f'Device: {device} | GPUs: {n_gpus} | AMP: {USE_AMP}')

Device: cuda | GPUs: 2 | AMP: True


In [2]:
os.system('pip install -q datasets huggingface_hub gradio scikit-image ')

0

In [3]:
import gc
import psutil

TOP_CATS = 20
TARGET_IMAGES =2500

SK_TX  = '/kaggle/input/datasets/hurairamuzammel/sketchy-data/256x256/sketch/tx_000000000000'
PH_TX  = '/kaggle/input/datasets/hurairamuzammel/sketchy-data/256x256/photo/tx_000000000000'

DATA_DIR = '/kaggle/working/data'
sk_dir   = os.path.join(DATA_DIR, 'sketches')
ph_dir   = os.path.join(DATA_DIR, 'photos')

SK_PKL   = '/kaggle/working/sk_cache.pkl'
PH_PKL   = '/kaggle/working/ph_cache.pkl'

for pkl in [SK_PKL, PH_PKL]:
    if os.path.exists(pkl):
        os.remove(pkl)

for d in [sk_dir, ph_dir]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

def progress(current, total, prefix='', bar_len=25):
    pct = current / total * 100
    bar = '█' * int(pct // (100 / bar_len)) + '░' * (bar_len - int(pct // (100 / bar_len)))
    print(f'\r  {prefix} [{bar}] {current}/{total} ({pct:.0f}%)', end='', flush=True)
    if current == total:
        print()

sk_categories = sorted([
    f for f in os.listdir(SK_TX)
    if os.path.isdir(os.path.join(SK_TX, f))
])

ph_categories = sorted([
    f for f in os.listdir(PH_TX)
    if os.path.isdir(os.path.join(PH_TX, f))
])

shared = sorted(set(sk_categories) & set(ph_categories))

top_shared = shared[:TOP_CATS]

print(f'Total shared categories : {len(shared)}')
print(f'Using top categories    : {len(top_shared)}')
print(f'Selected categories     : {top_shared}')

def find_images_in_folder(folder):
    result = subprocess.run(
        ['find', folder, '-type', 'f'],
        capture_output=True,
        text=True
    )

    files = sorted([
        f for f in result.stdout.strip().split('\n')
        if f.lower().endswith(
            ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
        )
    ])

    return files

sk_files_raw = []
ph_files_raw = []

per_cat_limit = TARGET_IMAGES // len(top_shared)

for cat in top_shared:

    sk_cat_files = find_images_in_folder(
        os.path.join(SK_TX, cat)
    )

    ph_cat_files = find_images_in_folder(
        os.path.join(PH_TX, cat)
    )

    n = min(
        len(sk_cat_files),
        len(ph_cat_files),
        per_cat_limit
    )

    sk_files_raw.extend(sk_cat_files[:n])
    ph_files_raw.extend(ph_cat_files[:n])

    print(
        f'{cat:20s} sketch:{len(sk_cat_files):4d} '
        f'photo:{len(ph_cat_files):4d} using:{n}'
    )

sk_files_raw = sk_files_raw[:TARGET_IMAGES]
ph_files_raw = ph_files_raw[:TARGET_IMAGES]

print(f'\nFinal sketch images : {len(sk_files_raw)}')
print(f'Final photo images  : {len(ph_files_raw)}')

def copy_single(args):
    p, dst, idx = args
    try:
        with Image.open(p) as im:
            img = im.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
            img.save(
                os.path.join(dst, f'{idx:05d}.jpg'),
                format='JPEG',
                quality=95
            )
        return True
    except Exception as e:
        print(f'  FAILED: {p} → {e}')
        return False

def copy_images_threaded(files, dst, label):
    total  = len(files)
    args   = [(p, dst, i) for i, p in enumerate(files)]
    done   = 0
    failed = 0

    print(f'Copying {total} {label} images...')

    with ThreadPoolExecutor(max_workers=8) as ex:
        for result in ex.map(copy_single, args):
            done   += result
            failed += (not result)
            progress(done + failed, total, prefix=label)

    print(f'{label} done — {done} copied, {failed} failed')
    return done

copy_images_threaded(sk_files_raw, sk_dir, 'Sketch')
copy_images_threaded(ph_files_raw, ph_dir, 'Photo')

total_sk = len(glob.glob(os.path.join(sk_dir, '*.jpg')))
total_ph = len(glob.glob(os.path.join(ph_dir, '*.jpg')))

print(f'\nSketch copied : {total_sk}')
print(f'Photo copied  : {total_ph}')

if total_sk == 0 or total_ph == 0:
    raise RuntimeError('No images copied.')

class RamSet(Dataset):

    def __init__(self, folder, aug=True):

        files = sorted(glob.glob(os.path.join(folder, '*.jpg')))

        print(f'Loading {len(files)} images into RAM...')

        t0   = time.time()
        imgs = []

        for i, p in enumerate(files):

            try:
                with Image.open(p) as im:
                    arr = np.array(
                        im.convert('RGB'),
                        dtype=np.uint8
                    )
                    imgs.append(
                        torch.from_numpy(arr)
                        .permute(2, 0, 1)
                    )
            except Exception:
                pass

            progress(i + 1, len(files), prefix='RAM load')

        self.data = torch.stack(imgs)
        self.aug  = aug

        print(
            f'Done in {time.time()-t0:.1f}s | '
            f'{self.data.nbytes/1024/1024:.0f} MB | '
            f'{len(self.data)} images'
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        img = self.data[idx]

        if self.aug and random.random() > 0.5:
            img = img.flip(-1)

        return img.float().div(127.5).sub(1.0)

def get_dataset(folder, aug, pkl_path):

    if os.path.exists(pkl_path):

        print(
            f'Loading cached dataset: {pkl_path} ...',
            end=' ',
            flush=True
        )

        t0 = time.time()

        with open(pkl_path, 'rb') as f:
            ds = pickle.load(f)

        print(
            f'done in {time.time()-t0:.1f}s | '
            f'{len(ds)} images'
        )

        return ds

    ds = RamSet(folder, aug=aug)

    print(
        f'Saving cache → {pkl_path} ...',
        end=' ',
        flush=True
    )

    with open(pkl_path, 'wb') as f:
        pickle.dump(ds, f)

    print('done')

    return ds

print('\nBuilding datasets:')

sk = get_dataset(sk_dir, aug=True, pkl_path=SK_PKL)
ph = get_dataset(ph_dir, aug=True, pkl_path=PH_PKL)

print(f'Sketch : {len(sk)}')
print(f'Photo  : {len(ph)}')

ld = dict(
    batch_size         = BATCH,
    shuffle            = False,
    num_workers        = 0,
    pin_memory         = True,
    drop_last          = True,
    persistent_workers = True,
)

sk_loader = DataLoader(sk, **ld)
ph_loader = DataLoader(ph, **ld)

print(f'Steps/epoch : {min(len(sk_loader), len(ph_loader))}')
print('Dataset ready')

Total shared categories : 125
Using top categories    : 20
Selected categories     : ['airplane', 'alarm_clock', 'ant', 'ape', 'apple', 'armor', 'axe', 'banana', 'bat', 'bear', 'bee', 'beetle', 'bell', 'bench', 'bicycle', 'blimp', 'bread', 'butterfly', 'cabin', 'camel']
airplane             sketch: 709 photo: 100 using:100
alarm_clock          sketch: 571 photo: 100 using:100
ant                  sketch: 563 photo: 100 using:100
ape                  sketch: 617 photo: 100 using:100
apple                sketch: 551 photo: 100 using:100
armor                sketch: 577 photo: 100 using:100
axe                  sketch: 602 photo: 100 using:100
banana               sketch: 635 photo: 100 using:100
bat                  sketch: 635 photo: 100 using:100
bear                 sketch: 722 photo: 100 using:100
bee                  sketch: 651 photo: 100 using:100
beetle               sketch: 549 photo: 100 using:100
bell                 sketch: 586 photo: 100 using:100
bench                sketch

In [4]:
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(ch, ch, 3),
            nn.InstanceNorm2d(ch),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(ch, ch, 3),
            nn.InstanceNorm2d(ch)
        )
    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ngf=64, n_res=N_RES):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, ngf, 7),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True)
        ]
        ch = ngf
        for _ in range(2):
            layers += [
                nn.Conv2d(ch, ch*2, 3, stride=2, padding=1),
                nn.InstanceNorm2d(ch*2),
                nn.ReLU(inplace=True)
            ]
            ch *= 2
        for _ in range(n_res):
            layers.append(ResBlock(ch))
        for _ in range(2):
            layers += [
                nn.ConvTranspose2d(ch, ch//2, 3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(ch//2),
                nn.ReLU(inplace=True)
            ]
            ch //= 2
        layers += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(ch, out_ch, 7),
            nn.Tanh()
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

class PatchDiscriminator(nn.Module):
    def __init__(self, in_ch=3, ndf=64):
        super().__init__()
        def block(ic, oc, norm=True):
            layers = [nn.Conv2d(ic, oc, 4, stride=2, padding=1)]
            if norm:
                layers.append(nn.InstanceNorm2d(oc))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        self.model = nn.Sequential(
            *block(in_ch, ndf, norm=False),
            *block(ndf, ndf*2),
            *block(ndf*2, ndf*4),
            nn.Conv2d(ndf*4, ndf*8, 4, padding=1),
            nn.InstanceNorm2d(ndf*8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, padding=1)
        )

    def forward(self, x):
        return self.model(x)

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.InstanceNorm2d) and m.weight is not None:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

G_AB = Generator().to(device)
G_BA = Generator().to(device)
D_A = PatchDiscriminator().to(device)
D_B = PatchDiscriminator().to(device)

for m in [G_AB, G_BA, D_A, D_B]:
    m.apply(init_weights)

if n_gpus > 1:
    G_AB = nn.DataParallel(G_AB)
    G_BA = nn.DataParallel(G_BA)
    D_A = nn.DataParallel(D_A)
    D_B = nn.DataParallel(D_B)

total_params = sum(p.numel() for p in G_AB.parameters())
print(f'Generator params: {total_params:,}')
print(f'Discriminator params: {sum(p.numel() for p in D_A.parameters()):,}')

Generator params: 7,837,699
Discriminator params: 2,764,737


In [5]:
# class RamSet(Dataset):
#     def __init__(self, folder, load_size, crop_size, aug=True, n=MAX_SAMPLES):
#         exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp')
#         files = []
#         for e in exts:
#             files += glob.glob(os.path.join(folder, '**', e), recursive=True)
#         random.shuffle(files)
#         files = files[:n]

#         self.crop_size = crop_size
#         self.aug = aug

#         print(f'  Pre-loading {len(files)} images into RAM...', end=' ', flush=True)
#         t0 = time.time()
#         imgs = []
#         for p in files:
#             try:
#                 img = Image.open(p).convert('RGB')
#                 if img.size != (load_size, load_size):
#                     img = img.resize((load_size, load_size), Image.LANCZOS)
#                 arr = np.array(img, dtype=np.uint8)
#                 imgs.append(torch.from_numpy(arr).permute(2, 0, 1))
#             except Exception:
#                 pass
#         self.data = torch.stack(imgs)
#         print(f'done in {time.time()-t0:.1f}s  |  {self.data.nbytes/1024/1024:.0f} MB')

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         img = self.data[idx]
#         if self.aug:
#             if img.shape[1] > self.crop_size:
#                 i = random.randint(0, img.shape[1] - self.crop_size)
#                 j = random.randint(0, img.shape[2] - self.crop_size)
#                 img = img[:, i:i+self.crop_size, j:j+self.crop_size]
#             if random.random() > 0.5:
#                 img = img.flip(-1)
#         return img.float().div(127.5).sub(1.0)

# pair = min(total_sk, total_ph, MAX_SAMPLES)
# print(f'Loading datasets (max {pair} each):')
# sk = RamSet(sk_dir, IMG_SIZE, IMG_SIZE, aug=True, n=pair)
# ph = RamSet(ph_dir, IMG_SIZE, IMG_SIZE, aug=True, n=pair)
# print(f'Sketch: {len(sk)}  Photo: {len(ph)}')

# ld = dict(
#     batch_size=BATCH,
#     shuffle=True,
#     num_workers=0,
#     pin_memory=torch.cuda.is_available(),
#     drop_last=True,
#  )
# sk_loader = DataLoader(sk, **ld)
# ph_loader = DataLoader(ph, **ld)
# print(f'Steps/epoch: {min(len(sk_loader), len(ph_loader))}')


In [6]:
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"Models already initialised — skipping duplicate cell.")
print(f"Generator params: {sum(p.numel() for p in G_AB.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D_A.parameters()):,}")


Models already initialised — skipping duplicate cell.
Generator params: 7,837,699
Discriminator params: 2,764,737


In [7]:
adv = nn.MSELoss()
cyc = nn.L1Loss()
idt = nn.L1Loss()

opt_g  = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()), lr=LR, betas=BETAS)
opt_da = torch.optim.Adam(D_A.parameters(), lr=LR, betas=BETAS)
opt_db = torch.optim.Adam(D_B.parameters(), lr=LR, betas=BETAS)

def lr_fn(epoch):
    cut = EPOCHS // 2
    if epoch < cut:
        return 1.0
    return max(0.0, 1.0 - (epoch - cut) / float(cut))

sch_g  = torch.optim.lr_scheduler.LambdaLR(opt_g,  lr_fn)
sch_da = torch.optim.lr_scheduler.LambdaLR(opt_da, lr_fn)
sch_db = torch.optim.lr_scheduler.LambdaLR(opt_db, lr_fn)

scaler = GradScaler() if USE_AMP else None

class Pool:
    def __init__(self, max_size=50):
        self.buf = []
        self.max = max_size

    def push_pop(self, imgs):
        ret = []
        for img in imgs.detach():
            img = img.unsqueeze(0)
            if len(self.buf) < self.max:
                self.buf.append(img.clone())
                ret.append(img)
            elif random.random() > 0.5:
                i = random.randint(0, self.max - 1)
                ret.append(self.buf[i].clone())
                self.buf[i] = img.clone()
            else:
                ret.append(img)
        return torch.cat(ret, 0)

pool_a = Pool()
pool_b = Pool()
print('Optimizers, schedulers, replay buffers ready.')
print(f'LR decay starts at epoch {EPOCHS//2}')


Optimizers, schedulers, replay buffers ready.
LR decay starts at epoch 12


In [8]:
import gc
loss = {'g': [], 'da': [], 'db': [], 'cyc': [], 'id': []}

def run_g(a, b):
    fa = G_AB(a)
    fb = G_BA(b)
    ra = G_BA(fb)
    rb = G_AB(fa)
    ones = torch.ones_like(D_B(fb))
    lg = adv(D_B(fb), ones) + adv(D_A(fa), ones)
    lc = (cyc(ra, a) + cyc(rb, b)) * LAMBDA_CYCLE
    if LAMBDA_ID > 0:
        ia = G_BA(a)
        ib = G_AB(b)
        li = (idt(ia, a) + idt(ib, b)) * LAMBDA_ID
    else:
        li = torch.tensor(0.0, device=device)
    return fa, fb, lg + lc + li, lc, li

def run_d(d, real, fake):
    ones = torch.ones_like(d(real))
    zeros = torch.zeros_like(ones)
    return (adv(d(real), ones) + adv(d(fake), zeros)) * 0.5

def train_ep(ep, bar=None):
    G_AB.train(); G_BA.train(); D_A.train(); D_B.train()
    g = da = db = cy = idv = 0.0
    steps = min(len(sk_loader), len(ph_loader))
    sk_it = iter(sk_loader)
    ph_it = iter(ph_loader)

    for step in range(steps):
        a = next(sk_it).to(device, non_blocking=True)
        b = next(ph_it).to(device, non_blocking=True)

        opt_g.zero_grad(set_to_none=True)
        if USE_AMP:
            with autocast():
                fa, fb, lg, lc, li = run_g(a, b)
            scaler.scale(lg).backward()
            scaler.step(opt_g)
        else:
            fa, fb, lg, lc, li = run_g(a, b)
            lg.backward()
            opt_g.step()

        fa_buf = pool_a.push_pop(fa)
        fb_buf = pool_b.push_pop(fb)
        del fa, fb

        opt_da.zero_grad(set_to_none=True)
        if USE_AMP:
            with autocast():
                l_da = run_d(D_A, a, fa_buf)
            scaler.scale(l_da).backward()
            scaler.step(opt_da)
        else:
            l_da = run_d(D_A, a, fa_buf)
            l_da.backward()
            opt_da.step()

        opt_db.zero_grad(set_to_none=True)
        if USE_AMP:
            with autocast():
                l_db = run_d(D_B, b, fb_buf)
            scaler.scale(l_db).backward()
            scaler.step(opt_db)
            scaler.update()
        else:
            l_db = run_d(D_B, b, fb_buf)
            l_db.backward()
            opt_db.step()

        del fa_buf, fb_buf

        g += lg.item()
        da += l_da.item()
        db += l_db.item()
        cy += lc.item()
        idv += li.item()

        if bar is not None:
            bar.set_postfix(
                G=f'{g/(step+1):.3f}',
                DA=f'{da/(step+1):.3f}',
                DB=f'{db/(step+1):.3f}',
                Cyc=f'{cy/(step+1):.3f}',
                refresh=False,
            )
            bar.update(1)

    return g/steps, da/steps, db/steps, cy/steps, idv/steps

print('Training function defined.')


Training function defined.


In [9]:
import gc
import psutil

adv_loss = nn.MSELoss()
cyc_loss = nn.L1Loss()
idt_loss = nn.L1Loss()

opt_g  = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()), lr=LR, betas=BETAS)
opt_da = torch.optim.Adam(D_A.parameters(), lr=LR, betas=BETAS)
opt_db = torch.optim.Adam(D_B.parameters(), lr=LR, betas=BETAS)

def lr_fn(epoch):
    cut = EPOCHS // 2
    if epoch < cut:
        return 1.0
    return max(0.0, 1.0 - (epoch - cut) / float(max(cut, 1)))

sch_g  = torch.optim.lr_scheduler.LambdaLR(opt_g,  lr_fn)
sch_da = torch.optim.lr_scheduler.LambdaLR(opt_da, lr_fn)
sch_db = torch.optim.lr_scheduler.LambdaLR(opt_db, lr_fn)

scaler = GradScaler() if USE_AMP else None

class Pool:
    def __init__(self, max_size=50):
        self.buf = []
        self.max = max_size

    def push_pop(self, imgs):
        ret = []
        for img in imgs.detach().cpu():
            img = img.unsqueeze(0)
            if len(self.buf) < self.max:
                self.buf.append(img.clone())
                ret.append(img)
            elif random.random() > 0.5:
                i = random.randint(0, self.max - 1)
                ret.append(self.buf[i].clone())
                self.buf[i] = img.clone()
            else:
                ret.append(img)
        return torch.cat(ret, 0).to(device, non_blocking=True)

pool_a = Pool()
pool_b = Pool()

def run_g(a, b):
    fa = G_AB(a)
    fb = G_BA(b)

    ra = G_BA(fa)
    rb = G_AB(fb)
    lc = (cyc_loss(ra, a) + cyc_loss(rb, b)) * LAMBDA_CYCLE
    del ra, rb

    d_fa = D_B(fa)
    d_fb = D_A(fb)
    lg = adv_loss(d_fa, torch.ones_like(d_fa)) + adv_loss(d_fb, torch.ones_like(d_fb))
    del d_fa, d_fb

    if LAMBDA_ID > 0:
        ia = G_BA(a)
        ib = G_AB(b)
        li = (idt_loss(ia, a) + idt_loss(ib, b)) * LAMBDA_ID
        del ia, ib
    else:
        li = torch.tensor(0.0, device=device)

    total = lg + lc + li
    del lg, lc, li
    return fa, fb, total

def run_d(d, real, fake):
    d_real = d(real)
    ones   = torch.ones_like(d_real)
    zeros  = torch.zeros_like(d_real)
    l      = (adv_loss(d_real, ones) + adv_loss(d(fake.detach()), zeros)) * 0.5
    del d_real, ones, zeros
    return l

def free_ram_gb():
    return psutil.virtual_memory().available / 1024**3

def free_vram_gb():
    if not torch.cuda.is_available():
        return float('inf')
    gc.collect()
    torch.cuda.empty_cache()
    free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)
    return free / 1024**3

def save_img(ep):
    G_AB.eval(); G_BA.eval()
    with torch.no_grad():
        a  = next(iter(sk_loader)).to(device)[:4]
        b  = next(iter(ph_loader)).to(device)[:4]
        fb = G_AB(a)
        fa = G_BA(b)
        ra = G_BA(fb)
        rb = G_AB(fa)
        imgs = torch.cat([a, fb, ra, b, fa, rb], 0)
        save_image(imgs * 0.5 + 0.5,
                   os.path.join(IMG_DIR, f'epoch_{ep:03d}.png'), nrow=4)
        del a, b, fb, fa, ra, rb, imgs
    torch.cuda.empty_cache()
    plt.close('all')
    G_AB.train(); G_BA.train()

def save_ckpt(ep):
    state = {
        'epoch' : ep,
        'G_AB'  : G_AB.state_dict() if not isinstance(G_AB, nn.DataParallel) else G_AB.module.state_dict(),
        'G_BA'  : G_BA.state_dict() if not isinstance(G_BA, nn.DataParallel) else G_BA.module.state_dict(),
        'D_A'   : D_A.state_dict()  if not isinstance(D_A,  nn.DataParallel) else D_A.module.state_dict(),
        'D_B'   : D_B.state_dict()  if not isinstance(D_B,  nn.DataParallel) else D_B.module.state_dict(),
        'opt_G' : opt_g.state_dict(),
        'opt_DA': opt_da.state_dict(),
        'opt_DB': opt_db.state_dict(),
        'hist'  : loss,
    }
    torch.save(state, os.path.join(CKPT_DIR, f'ckpt_ep{ep:03d}.pth'))
    del state
    gc.collect()

def train_ep(ep):
    G_AB.train(); G_BA.train(); D_A.train(); D_B.train()

    g = da = db = cy_sum = id_sum = 0.0
    steps = min(len(sk_loader), len(ph_loader))

    sk_it = iter(sk_loader)
    ph_it = iter(ph_loader)

    for step in range(steps):
        a = next(sk_it).to(device, non_blocking=True)
        b = next(ph_it).to(device, non_blocking=True)

        opt_g.zero_grad(set_to_none=True)

        if USE_AMP:
            with autocast():
                fa, fb, lg = run_g(a, b)
            scaler.scale(lg).backward()
            scaler.step(opt_g)
        else:
            fa, fb, lg = run_g(a, b)
            lg.backward()
            opt_g.step()

        g += lg.item()
        del lg

        fa_buf = pool_a.push_pop(fa)
        fb_buf = pool_b.push_pop(fb)
        del fa, fb

        opt_da.zero_grad(set_to_none=True)

        if USE_AMP:
            with autocast():
                l_da = run_d(D_A, a, fa_buf)
            scaler.scale(l_da).backward()
            scaler.step(opt_da)
        else:
            l_da = run_d(D_A, a, fa_buf)
            l_da.backward()
            opt_da.step()

        da += l_da.item()
        del l_da, fa_buf

        opt_db.zero_grad(set_to_none=True)

        if USE_AMP:
            with autocast():
                l_db = run_d(D_B, b, fb_buf)
            scaler.scale(l_db).backward()
            scaler.step(opt_db)
            scaler.update()
        else:
            l_db = run_d(D_B, b, fb_buf)
            l_db.backward()
            opt_db.step()

        db += l_db.item()
        del l_db, fb_buf, a, b

        if step % 50 == 0:
            torch.cuda.empty_cache()

        if (step + 1) % 50 == 0 or step == steps - 1:
            print(f'\r  Ep {ep:03d}/{EPOCHS} step {step+1}/{steps} '
                  f'| G:{g/(step+1):.3f} DA:{da/(step+1):.3f} DB:{db/(step+1):.3f}',
                  end='', flush=True)

    print()
    return g/steps, da/steps, db/steps

loss = {'g': [], 'da': [], 'db': []}

print(f'Training starting — {EPOCHS} epochs | {min(len(sk_loader), len(ph_loader))} steps/epoch')
print(f'LR decay starts at epoch {EPOCHS//2}')
print('-' * 60)

total_start = time.time()
skipped_eps = []

for ep in range(1, EPOCHS + 1):

    free_ram = free_ram_gb()
    free_vram = free_vram_gb()

    print(f'Ep {ep:03d} — RAM free: {free_ram:.1f} GB | VRAM free: {free_vram:.2f} GB')

    if free_vram < 0.5:
        print(f'  !! Skipping ep {ep} — only {free_vram:.2f} GB VRAM free')
        skipped_eps.append(ep)
        gc.collect()
        torch.cuda.empty_cache()
        continue

    t0 = time.time()

    lg, lda, ldb = train_ep(ep)

    loss['g'].append(lg)
    loss['da'].append(lda)
    loss['db'].append(ldb)

    sch_g.step(); sch_da.step(); sch_db.step()

    elapsed = time.time() - t0
    eta     = elapsed * (EPOCHS - ep)
    eta_str = f'{eta/3600:.1f}h' if eta > 3600 else f'{eta/60:.1f}m'

    print(f'Ep {ep:03d}/{EPOCHS} | G:{lg:.4f} DA:{lda:.4f} DB:{ldb:.4f} '
          f'| {elapsed:.1f}s/ep | ETA:{eta_str}')

    if ep % 5 == 0 or ep == EPOCHS:
        save_img(ep)
        save_ckpt(ep)
        print(f'  ✓ Checkpoint + sample saved (ep {ep})')

    gc.collect()
    torch.cuda.empty_cache()
    plt.close('all')

    ram_after  = free_ram_gb()
    vram_after = free_vram_gb()

    print(f'  RAM free after: {ram_after:.1f} GB | '
          f'VRAM alloc: {torch.cuda.memory_allocated()/1024**3:.2f} GB | '
          f'VRAM free: {vram_after:.2f} GB')

total_elapsed = time.time() - total_start

print('=' * 60)
print(f'Training complete in {total_elapsed/3600:.2f}h')

if skipped_eps:
    print(f'Skipped epochs: {skipped_eps}')

print(f'Final  G:{loss["g"][-1]:.4f}  DA:{loss["da"][-1]:.4f}  DB:{loss["db"][-1]:.4f}')

final_state = {
    'epoch': EPOCHS,
    'G_AB' : G_AB.state_dict() if not isinstance(G_AB, nn.DataParallel) else G_AB.module.state_dict(),
    'G_BA' : G_BA.state_dict() if not isinstance(G_BA, nn.DataParallel) else G_BA.module.state_dict(),
    'D_A'  : D_A.state_dict()  if not isinstance(D_A,  nn.DataParallel) else D_A.module.state_dict(),
    'D_B'  : D_B.state_dict()  if not isinstance(D_B,  nn.DataParallel) else D_B.module.state_dict(),
    'hist' : loss,
}

torch.save(final_state, os.path.join(OUTPUT_DIR, 'cyclegan_final.pth'))

del final_state
gc.collect()
torch.cuda.empty_cache()

print(f'Final model saved → {OUTPUT_DIR}/cyclegan_final.pth')

Training starting — 25 epochs | 250 steps/epoch
LR decay starts at epoch 12
------------------------------------------------------------
Ep 001 — RAM free: 29.2 GB | VRAM free: 14.46 GB
  Ep 001/25 step 250/250 | G:4.578 DA:0.248 DB:0.216
Ep 001/25 | G:4.5777 DA:0.2475 DB:0.2157 | 112.3s/ep | ETA:44.9m
  RAM free after: 26.9 GB | VRAM alloc: 0.41 GB | VRAM free: 13.89 GB
Ep 002 — RAM free: 26.9 GB | VRAM free: 13.89 GB
  Ep 002/25 step 250/250 | G:3.248 DA:0.048 DB:0.047
Ep 002/25 | G:3.2484 DA:0.0481 DB:0.0467 | 109.3s/ep | ETA:41.9m
  RAM free after: 25.8 GB | VRAM alloc: 0.41 GB | VRAM free: 13.82 GB
Ep 003 — RAM free: 25.8 GB | VRAM free: 13.82 GB
  Ep 003/25 step 250/250 | G:2.795 DA:0.022 DB:0.024
Ep 003/25 | G:2.7952 DA:0.0222 DB:0.0241 | 109.8s/ep | ETA:40.3m
  RAM free after: 24.7 GB | VRAM alloc: 0.41 GB | VRAM free: 13.85 GB
Ep 004 — RAM free: 24.7 GB | VRAM free: 13.85 GB
  Ep 004/25 step 250/250 | G:2.675 DA:0.031 DB:0.021
Ep 004/25 | G:2.6754 DA:0.0308 DB:0.0205 | 109.7s/

In [12]:
import os
import numpy as np
import matplotlib.pyplot as plt

epochs_done = len(loss.get('g', []))

if epochs_done == 0:
    print('No training data to plot')

else:

    eps = range(1, epochs_done + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    if 'g' in loss:
        axes[0].plot(eps, loss['g'])
    axes[0].set_title('Generator Loss vs Epochs')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True)

    if 'da' in loss:
        axes[1].plot(eps, loss['da'])

    if 'db' in loss:
        axes[1].plot(eps, loss['db'])

    axes[1].set_title('Discriminator Loss vs Epochs')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True)

    if 'cyc' in loss:
        axes[2].plot(eps, loss['cyc'])

    if 'id' in loss:
        axes[2].plot(eps, loss['id'])

    axes[2].set_title('Cycle & Identity Loss')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Loss')
    axes[2].grid(True)

    plt.tight_layout()

    save_path = os.path.join(
        OUTPUT_DIR,
        'training_curves.png'
    )

    plt.savefig(save_path, dpi=150)

    plt.show()

    print('Training curves saved to:')
    print(save_path)

Training curves saved to:
/kaggle/working/training_curves.png


In [14]:
import matplotlib.pyplot as plt



import numpy as np

def to_np(t):

    img = t.detach().cpu()

    if img.dim() == 4:
        img = img[0]

    img = img * 0.5 + 0.5
    img = img.clamp(0, 1)

    img = img.permute(1, 2, 0).numpy()

    img = (img * 255).astype(np.uint8)

    return img
    
G_AB.eval()
G_BA.eval()

rows = 4
cols = 6

fig, axes = plt.subplots(rows, cols, figsize=(15, 10))

with torch.no_grad():

    sk_it = iter(sk_loader)
    ph_it = iter(ph_loader)

    for row in range(rows):

        real_A = next(sk_it).to(device)[:1]
        real_B = next(ph_it).to(device)[:1]

        fake_B = G_AB(real_A)
        fake_A = G_BA(real_B)

        rec_A = G_BA(fake_B)
        rec_B = G_AB(fake_A)

        imgs_row = [
            real_A,
            fake_B,
            rec_A,
            real_B,
            fake_A,
            rec_B
        ]

        for col, img_t in enumerate(imgs_row):

            axes[row, col].imshow(
                to_np(img_t)
            )

            axes[row, col].axis('off')

plt.tight_layout()

save_path = os.path.join(
    OUTPUT_DIR,
    'sample_results.png'
)

plt.savefig(save_path, dpi=150)

plt.show()

print('Samples saved to:')
print(save_path)

Samples saved to:
/kaggle/working/sample_results.png


In [15]:
import gradio as gr

g_ab = Generator().to(device)
g_ba = Generator().to(device)

g_ab_sd = G_AB.module.state_dict() if isinstance(G_AB, nn.DataParallel) else G_AB.state_dict()
g_ba_sd = G_BA.module.state_dict() if isinstance(G_BA, nn.DataParallel) else G_BA.state_dict()
g_ab.load_state_dict(g_ab_sd)
g_ba.load_state_dict(g_ba_sd)
g_ab.eval(); g_ba.eval()

def go(img, d):
    if img is None:
        return None
    if isinstance(img, np.ndarray):
        img = Image.fromarray(img.astype(np.uint8)).convert('RGB')
    x = tfm(img).unsqueeze(0).to(device)
    with torch.no_grad():
        if d == 'Sketch -> Photo':
            out = g_ab(x)
        else:
            out = g_ba(x)
    return Image.fromarray(to_np(out))

with gr.Blocks(title='CycleGAN Sketch-Photo Translation') as demo:
    gr.Markdown('## CycleGAN: Sketch Photo Domain Translation')
    gr.Markdown('Upload an image and select translation direction.')
    with gr.Row():
        with gr.Column():
            inp = gr.Image(label='Input Image', type='numpy')
            d = gr.Radio(
                choices=['Sketch -> Photo', 'Photo -> Sketch'],
                value='Sketch -> Photo',
                label='Translation Direction'
            )
            btn = gr.Button('Translate', variant='primary')
        with gr.Column():
            out = gr.Image(label='Translated Output')
    btn.click(fn=go, inputs=[inp, d], outputs=out)
    gr.Examples(
        examples=[[sk_files[0], 'Sketch -> Photo'], [ph_files[0], 'Photo -> Sketch']] if sk_files and ph_files else [],
        inputs=[inp, d]
    )

demo.launch(share=True, quiet=True)
print('Gradio app launched.')

* Running on public URL: https://00adeebdf2b68c8930.gradio.live


Gradio app launched.


In [20]:
import numpy as np
import json
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

def to_np(t):

    img = t.detach().cpu()

    if img.dim() == 4:
        img = img[0]

    img = img * 0.5 + 0.5
    img = img.clamp(0, 1)

    img = img.permute(1, 2, 0).numpy()

    return img


def score(real, fake):

    r = to_np(real)
    f = to_np(fake)

    s = ssim(
        r,
        f,
        channel_axis=2,
        data_range=1.0
    )

    p = psnr(
        r,
        f,
        data_range=1.0
    )

    return s, p


G_AB.eval()
G_BA.eval()

ssim_ab_list = []
psnr_ab_list = []

ssim_ba_list = []
psnr_ba_list = []

with torch.no_grad():

    for i, (batch_A, batch_B) in enumerate(
        zip(sk_loader, ph_loader)
    ):

        if i >= 10:
            break

        real_A = batch_A.to(device)
        real_B = batch_B.to(device)

        fake_B = G_AB(real_A)
        fake_A = G_BA(real_B)

        rec_A = G_BA(fake_B)
        rec_B = G_AB(fake_A)

        for j in range(real_A.size(0)):

            s, p = score(real_A[j], rec_A[j])
            ssim_ab_list.append(s)
            psnr_ab_list.append(p)

            s, p = score(real_B[j], rec_B[j])
            ssim_ba_list.append(s)
            psnr_ba_list.append(p)


metrics = {

    'sketch_cycle_ssim':
        float(np.mean(ssim_ab_list)),

    'sketch_cycle_psnr':
        float(np.mean(psnr_ab_list)),

    'photo_cycle_ssim':
        float(np.mean(ssim_ba_list)),

    'photo_cycle_psnr':
        float(np.mean(psnr_ba_list))
}

metrics_path = os.path.join(
    OUTPUT_DIR,
    'metrics.json'
)

with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print('\nFinal Metrics Summary:')
print(f'  Sketch Cycle SSIM : {metrics["sketch_cycle_ssim"]:.4f}')
print(f'  Sketch Cycle PSNR : {metrics["sketch_cycle_psnr"]:.2f} dB')
print(f'  Photo  Cycle SSIM : {metrics["photo_cycle_ssim"]:.4f}')
print(f'  Photo  Cycle PSNR : {metrics["photo_cycle_psnr"]:.2f} dB')

print('\nMetrics saved to:')
print(metrics_path)


Final Metrics Summary:
  Sketch Cycle SSIM : 0.9913
  Sketch Cycle PSNR : 29.91 dB
  Photo  Cycle SSIM : 0.8466
  Photo  Cycle PSNR : 24.49 dB

Metrics saved to:
/kaggle/working/metrics.json
